# 19 — Consolidar os resultados e demonstrar os dois modelos

Os ajustes 16–18 não produziram um novo modelo que cumprisse os seis critérios.
Agora a entrega é uma **comparação visual e documentação dos resultados**.

- Consolida baseline, candidato 10 e checkpoints dos experimentos 16/18.
- Registra o resultado da grade de 49 limiares do 17.
- Gera uma demonstração comparativa em até 24 imagens das validações,
  com referência anotada, previsões, mensagens V2, acertos e falhas.
- Produz tabelas e texto-base para o capítulo experimental do TCC.
- Fixa o protocolo da avaliação posterior na reserva externa de 300 imagens.

**Não há novo treino, nova inferência ou consulta à reserva nesta etapa.**
As previsões são reaproveitadas dos pacotes autenticados. A galeria é ilustrativa;
as métricas completas usam as 456 Home VAL e 1.000 externas já conhecidas.


## 1. Executar como novo notebook no Kaggle

Importe este arquivo como **novo notebook**. Selecione **CPU**, ative **Internet**
e habilite o Secret existente **GDRIVE_RCLONE_CONFIG_B64**. Use
**Save Version → Save & Run All**. Não precisa alterar caminhos ou parâmetros.

**Estimativa: 5–15 minutos em CPU**, principalmente para baixar os dois ZIPs de
imagens e salvar os arquivos. Downloads lentos podem aumentar esse tempo.
Não é preciso reservar GPU. Dos ZIPs, só serão extraídas as imagens selecionadas
das validações; imagens/labels de treino, TEST e reserva não serão abertos.


In [ ]:
%pip install -q numpy pillow pyyaml


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
import sys,json,hashlib,shutil,platform,time,gzip,zipfile
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import PIL
from IPython.display import display,FileLink
BASE_ID='ad3fce778297bc91e5a196cb'
PASTA_LOCAL=Path('/tmp/tcc_consolidacao_19')/BASE_ID
PASTA_BOOT=Path('/kaggle/working/resultados_tcc')/('consolidacao19_preparacao_'+BASE_ID)
PASTA_LOCAL.mkdir(parents=True,exist_ok=True);PASTA_BOOT.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS=PASTA_BOOT;NOME_EXPERIMENTO=PASTA_BOOT.name
DRIVE=None
INICIO=time.monotonic();ULTIMA_ETAPA=INICIO;TEMPOS=[]
AMBIENTE={'python':platform.python_version(),'numpy':np.__version__,'pillow':PIL.__version__,'execucao':'CPU; previsoes congeladas'}
print('Notebook 19: consolidação, demonstração e protocolo posterior. CPU suficiente.')
print('Previsões reaproveitadas. Nenhum treino ou inferência será iniciado.')


## 2. Conferir o código e a política V2

Os módulos estão embutidos e autenticados. A política e o associador são os mesmos
dos resultados anteriores. O fluxo não importa Torch ou Ultralytics e não carrega
pesos para gerar novas previsões.


In [ ]:
EXPECTED={'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f', 'politica': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'associador': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d', 'home_zip': 'ada7fbee37b83f673b1b76bc57643d1fe4648941e0e8a7b895282216aed47676', 'zip_novo': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'dataset_novo': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'regra_12_sha256': '04c6287c2c4adb8c43baf28c10acbaa1a599198b905d48808a8f5f67f39a4909', 'pacote_16': 'cd9acf4eba29bb28dee20465ec2503b72cc65302f3d7c8377dd442455a4d16fa', 'pacote_17': '0ef3ce66a70216c641176a8c682e74b7a08c758f5d98ccb1ebd3db31943fde93', 'pasta_16': 'resultados/controle16_44a814e0a728085209af51a3', 'pasta_17': 'resultados/limiares17_6f4aa31baf57035ce481a469', 'pacote_18': 'f3ea533498d5af59f6bb94a919e71c4a2e8970aae488802d5418e584e19b07d6', 'contexto_18': '129a908360b9bf239017b45e003a12753aa17f44b672188a94550a9e5bce2953', 'pasta_18': 'resultados/fontes18_129a908360b9bf239017b45e', 'pasta_validacao': 'datasets/validacao_independente_11_v1/695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'arquivo_validacao': 'validacao_independente_11_695b445e15c9.zip'}
MODULOS_EMBUTIDOS={'conf14_data.py': '"""Entradas autenticadas do diagnóstico 14; extrai somente Home VAL."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image\nimport diag11_core as core\n\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nBOOLS = {\'True\':True, \'False\':False}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef parse_events(raw):\n    result = rows(raw)\n    for r in result:\n        r[\'classe_id\'] = int(r[\'classe_id\'])\n        for field in [\'bbox_ref\',\'bbox_pred\']:\n            r[field] = json.loads(r[field]) if r[field] else None\n        for field in [\'ha_alerta_ref\',\'ha_alerta_pred\']:\n            r[field] = BOOLS[r[field]]\n        r[\'confianca\'] = float(r[\'confianca\']) if r[\'confianca\'] else None\n    return result\n\ndef gt_key(r):\n    return (r[\'id\'],int(r[\'classe_id\']),tuple(round(float(v),6) for v in r[\'bbox_ref\']))\n\ndef gt_map(events):\n    result = {}\n    for r in events:\n        if r[\'bbox_ref\'] is None: continue\n        key = gt_key(r)\n        if key in result: raise ValueError(\'Objeto GT repetido nos registros.\')\n        result[key] = r\n    return result\n\ndef load_source12(path, expected, output=None):\n    if core.sha256_file(path)!=expected[\'pacote_12\']:\n        raise RuntimeError(\'O pacote 12 mudou em relação ao recebido.\')\n    bundle = {\'evaluations\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):\n            raise RuntimeError(\'Pacote 12 com membros repetidos.\')\n        for name in z.namelist(): core.safe_member(name)\n        if z.testzip() is not None: raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        cfg = json.loads(z.read(\'configuracao_experimento_12.json\'))\n        summary = json.loads(z.read(\'resumo_experimento_12.json\'))\n        if not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\'] or summary[\'test_oi20k_inferido\']:\n            raise RuntimeError(\'Execução 12 incompatível com o protocolo.\')\n        if cfg[\'expected\'][\'candidato\']!=expected[\'candidato\'] or cfg[\'expected\'][\'politica\']!=expected[\'politica\'] or summary[\'dataset_sha256\']!=expected[\'dataset_novo\']:\n            raise RuntimeError(\'Referências ou política diferentes do experimento 12.\')\n        if cfg[\'codigo_embutido_sha256\']!=expected[\'codigo_12\']:\n            raise RuntimeError(\'Código do experimento 12 diferente do recebido.\')\n        for name, digest in expected[\'codigo_12\'].items():\n            if hashlib.sha256(z.read(\'codigo/\'+name)).hexdigest()!=digest:\n                raise RuntimeError(\'Código alterado no pacote 12: \'+name)\n        rule_raw = z.read(\'regra_selecao_congelada.json\')\n        if hashlib.sha256(rule_raw).hexdigest()!=cfg[\'regra_sha256\']:\n            raise RuntimeError(\'A regra original do experimento 12 foi alterada.\')\n        selection = json.loads(z.read(\'selecao_checkpoints.json\'))\n        checkpoint = next(e for e in selection[\'assessed\'] if e[\'id\']==\'B_home_replay_e03\')\n        if checkpoint[\'weight_sha256\']!=expected[\'B_e03\']:\n            raise RuntimeError(\'Checkpoint B_e03 diferente do esperado.\')\n        for model,prefix in [(\'baseline\',\'referencias/baseline\'),(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n            bundle[\'evaluations\'][model] = {}\n            for cohort in COHORTS:\n                base=prefix+\'/\'+cohort+\'/\'\n                members={name:z.read(base+name) for name in [\'metricas.json\',\'alertas_por_imagem.csv\',\'avaliacao_objetos.csv\']}\n                metrics=json.loads(members[\'metricas.json\'])\n                images=rows(members[\'alertas_por_imagem.csv\'])\n                for r in images:\n                    for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']:\n                        r[key]=int(r[key])\n                if len({r[\'id\'] for r in images})!=len(images) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou quantidades inconsistentes nos registros 12.\')\n                for field, n in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[field] for r in images)!=n: raise RuntimeError(\'Contagens inconsistentes no pacote 12.\')\n                events=parse_events(members[\'avaliacao_objetos.csv\'])\n                gt_map(events)\n                bundle[\'evaluations\'][model][cohort]={\'summary\':metrics,\'images\':images,\'events\':events}\n                if output is not None:\n                    dest=Path(output)/\'referencias_12\'/model/cohort\n                    dest.mkdir(parents=True,exist_ok=True)\n                    for name, raw in members.items(): (dest/name).write_bytes(raw)\n        for cohort in COHORTS:\n            target=gt_map(bundle[\'evaluations\'][\'B_e03\'][cohort][\'events\'])\n            for model in [\'baseline\',\'candidato_10\']:\n                source=gt_map(bundle[\'evaluations\'][model][cohort][\'events\'])\n                if source.keys()!=target.keys(): raise RuntimeError(\'Modelos do 12 usam referências GT diferentes.\')\n                if any(source[k][\'ha_alerta_ref\']!=target[k][\'ha_alerta_ref\'] or source[k][\'prioridade_ref\']!=target[k][\'prioridade_ref\'] for k in target):\n                    raise RuntimeError(\'Referências de prioridade diferentes entre modelos.\')\n        bundle.update(config=cfg,summary=summary,rule=json.loads(rule_raw),\n                      audit=json.loads(z.read(\'auditoria_dados.json\')))\n    return bundle\n\ndef extract_home_val(path, destination, expected_zip, expected_test, expected_policy):\n    if core.sha256_file(path)!=expected_zip: raise RuntimeError(\'ZIP Home alterado.\')\n    destination=Path(destination)\n    records=[];frozen=[]\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())): raise RuntimeError(\'Membros repetidos no Home.\')\n        for name in z.namelist():core.safe_member(name)\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if (manifest[\'test\'][\'sha256\']!=expected_test or manifest[\'test\'][\'imagens\']!=403 or\n            {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES))):\n            raise RuntimeError(\'Manifesto Home incompatível.\')\n        root=manifest[\'dataset_dir\']\n        items={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        if len(items)!=len(manifest[\'arquivos\']): raise RuntimeError(\'Manifesto Home repetido.\')\n        train_hash=set();val_hash=set();all_hash=set();counts=Counter()\n        for name,item in items.items():\n            is_image=name.startswith(root+\'/images/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}\n            if not is_image:continue\n            all_hash.add(item[\'sha256\'])\n            if name.startswith(root+\'/images/train/raw_train/\'):\n                counts[\'train\']+=1;train_hash.add(item[\'sha256\'])\n            elif name.startswith(root+\'/images/val/raw_train/\'):\n                counts[\'val\']+=1;val_hash.add(item[\'sha256\'])\n        if counts!={\'train\':1824,\'val\':456} or train_hash&val_hash:\n            raise RuntimeError(\'A divisão atual de Home mudou ou tem bytes repetidos entre TRAIN e VAL.\')\n        for name in z.namelist():\n            if not(name.startswith(\'codigo/\') and name.endswith(\'.py\')):continue\n            raw=z.read(name)\n            if name in items and hashlib.sha256(raw).hexdigest()!=items[name][\'sha256\']:\n                raise RuntimeError(\'Código Home alterado.\')\n            out=destination/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n            frozen.append((out,hashlib.sha256(raw).hexdigest()))\n        for name,item in sorted(items.items()):\n            if not(name.startswith(root+\'/images/val/raw_train/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}):continue\n            label=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            if label not in items:raise RuntimeError(\'Label Home ausente.\')\n            paths={}\n            for field,member in [(\'image\',name),(\'label\',label)]:\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=items[member][\'sha256\']:raise RuntimeError(\'Home alterado: \'+member)\n                out=destination/member;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n                paths[field]=out;frozen.append((out,items[member][\'sha256\']))\n            with Image.open(paths[\'image\']) as im:\n                width,height=im.size;im.verify()\n            records.append({\'id\':\'home:\'+name,\'name\':name.split(\'/raw_train/\')[-1],**paths,\n                \'role\':\'home_val\',\'coorte\':\'home_val_historica\',\'width\':width,\'height\':height,\n                \'known_mask\':core.ALL_KNOWN,\'sha256\':item[\'sha256\'],\'label_sha256\':items[label][\'sha256\'],\n                \'gt\':core.parse_yolo(paths[\'label\'].read_bytes(),width,height,source=label)})\n    if len(records)!=456 or core.sha256_file(destination/\'codigo/src/politica_alertas.py\')!=expected_policy:\n        raise RuntimeError(\'Quantidade Home VAL ou política V2 incompatível.\')\n    audit={\'home_train_manifesto\':1824,\'home_val_extraida\':456,\'home_train_extraida\':False,\n           \'home_test_extraido\':False,\'test_home_manifesto\':403,\'intersecao_bytes_train_val\':0,\n           \'historico_original_baseline_nao_comprovado_pela_separacao_atual\':True}\n    return records,all_hash,frozen,audit\n\ndef validate_cohorts(cohorts,bundle,reserved_ids):\n    if set(cohorts)!=set(COHORTS) or {k:len(v) for k,v in cohorts.items()}!={\'home_val_historica\':456,\'openimages_val_nova\':1000}:\n        raise ValueError(\'Coortes diferentes das duas validações congeladas.\')\n    for cohort,records in cohorts.items():\n        ids={r[\'id\'] for r in records}\n        if len(ids)!=len(records) or ids&set(reserved_ids):raise ValueError(\'ID repetido ou reservado na avaliação.\')\n        expected_role=\'home_val\' if cohort==\'home_val_historica\' else \'val_nova\'\n        if any(r[\'role\']!=expected_role for r in records):raise ValueError(\'Papel incompatível com a coorte de validação.\')\n        old=bundle[\'evaluations\'][\'B_e03\'][cohort]\n        if ids!={r[\'id\'] for r in old[\'images\']}:raise ValueError(\'IDs diferentes dos avaliados no 12.\')\n        actual={}\n        for r in records:\n            for g in r[\'gt\']:\n                if r[\'known_mask\']&(1<<g[\'classe_id\']):\n                    k=(r[\'id\'],g[\'classe_id\'],tuple(round(float(x),6) for x in g[\'bbox\']))\n                    if k in actual:raise ValueError(\'GT repetido na entrada atual.\')\n                    actual[k]=True\n        if actual.keys()!=gt_map(old[\'events\']).keys():raise ValueError(\'Caixas avaliáveis diferentes das referências originais.\')\n    return True\n', 'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"Checkpoint / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'cons19_demo.py': '"""Galeria local com a mesma imagem, GT e previsões dos dois modelos."""\nfrom __future__ import annotations\nimport base64,hashlib,html,json,mimetypes,zipfile\nfrom pathlib import Path\nfrom PIL import Image\nimport diag11_core as core\nimport cons19_protocol as protocol\n\nCATEGORY_LABELS={\'mais_tp_candidato10\':\'Candidato 10 recupera mais alertas\',\n    \'mais_tp_baseline\':\'Baseline recupera mais alertas\',\'omissao_alta\':\'Omissão de alta prioridade\',\n    \'falso_alerta\':\'Presença de falso alerta\',\'acerto_sem_erro\':\'Acertos sem FN/FP nas classes avaliáveis\',\n    \'sem_alerta_referencia\':\'Sem alerta de referência nas classes avaliáveis\',\'complemento\':\'Complemento por ordem fixa\'}\n\ndef select(records,evaluations,spec=protocol.SPEC):\n    selected=[];manifest=[]\n    for cohort in protocol.COHORTS:\n        mapping={m:{r[\'id\']:r for r in evaluations[m][cohort][\'images\']} for m in protocol.MODELS}\n        candidates=[r for r in records if r[\'coorte\']==cohort]\n        if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in candidates):raise ValueError(\'Papel protegido na galeria.\')\n        used=set();chosen=[]\n        def eligible(category,r):\n            a,b=[mapping[m][r[\'id\']] for m in protocol.MODELS]\n            if category==\'mais_tp_candidato10\':return b[\'tp_alerta\']>a[\'tp_alerta\']\n            if category==\'mais_tp_baseline\':return a[\'tp_alerta\']>b[\'tp_alerta\']\n            if category==\'omissao_alta\':return max(a[\'alta_sem_alerta\'],b[\'alta_sem_alerta\'])>0\n            if category==\'falso_alerta\':return max(a[\'fp_alerta\'],b[\'fp_alerta\'])>0\n            if category==\'acerto_sem_erro\':return all(v[\'tp_alerta\']>0 and v[\'fn_alerta\']==v[\'fp_alerta\']==0 for v in [a,b])\n            if category==\'sem_alerta_referencia\':return a[\'tp_alerta\']+a[\'fn_alerta\']==0\n            return True\n        order=lambda r:hashlib.sha256((spec[\'demo_seed\']+\'|\'+r[\'id\']).encode()).hexdigest()\n        for category in list(spec[\'demo_categories\'])+[\'complemento\']:\n            limit=spec[\'demo_images_per_cohort\']-len(chosen) if category==\'complemento\' else spec[\'demo_per_category\']\n            pool=sorted([r for r in candidates if r[\'id\'] not in used and eligible(category,r)],key=order)\n            for record in pool[:max(0,limit)]:\n                if len(chosen)>=spec[\'demo_images_per_cohort\']:break\n                used.add(record[\'id\']);chosen.append(record)\n                manifest.append({\'coorte\':cohort,\'id\':record[\'id\'],\'image_sha256\':record[\'sha256\'],\n                    \'categoria\':category,\'ordem_hash\':order(record),\'ilustrativa_sem_peso_nas_metricas\':True})\n        selected+=chosen\n    if len({r[\'id\'] for r in selected})!=len(selected):raise ValueError(\'Imagem repetida na seleção.\')\n    return selected,manifest\n\ndef image_member(record):\n    if record[\'role\']==\'home_val\' and record[\'coorte\']==\'home_val_historica\':\n        name=record[\'id\'].removeprefix(\'home:\')\n        if not name.startswith(\'homeobjects_yolo/images/val/raw_train/\'):raise ValueError(\'Imagem Home fora de VAL.\')\n        return name\n    if record[\'role\']==\'val_nova\' and record[\'coorte\']==\'openimages_val_nova\':\n        name=record[\'name\']\n        if Path(name).name!=name:raise ValueError(\'Nome externo inválido.\')\n        return \'images/val_nova/\'+name\n    raise ValueError(\'A galeria só aceita as validações de desenvolvimento.\')\n\ndef extract_images(selected,archives,expected,output):\n    output=Path(output);paths={};receipts=[]\n    for cohort in protocol.COHORTS:\n        rows=[r for r in selected if r[\'coorte\']==cohort]\n        if not rows:continue\n        archive=Path(archives[cohort]);key=\'home_zip\' if cohort==\'home_val_historica\' else \'zip_novo\'\n        if core.sha256_file(archive)!=expected[key]:raise ValueError(\'ZIP de imagens alterado: \'+cohort)\n        with zipfile.ZipFile(archive) as z:\n            if len(z.namelist())!=len(set(z.namelist())):raise ValueError(\'ZIP com membros repetidos.\')\n            for name in z.namelist():core.safe_member(name)\n            for r in rows:\n                member=image_member(r);core.safe_member(member)\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=r[\'sha256\']:raise ValueError(\'Imagem da galeria difere do cache: \'+r[\'id\'])\n                ext=Path(member).suffix.lower();path=output/\'demonstracao/imagens\'/(hashlib.sha256(r[\'id\'].encode()).hexdigest()[:24]+ext)\n                path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)\n                with Image.open(path) as image:\n                    if image.size!=(r[\'width\'],r[\'height\']):raise ValueError(\'Dimensões da galeria divergentes.\')\n                    image.verify()\n                paths[r[\'id\']]=path\n                receipts.append({\'id\':r[\'id\'],\'role\':r[\'role\'],\'member\':member,\'sha256\':r[\'sha256\'],\'relative\':str(path.relative_to(output))})\n    if set(paths)!={r[\'id\'] for r in selected}:raise ValueError(\'Galeria sem alguma imagem.\')\n    core.save_json(output/\'imagens_demo_extraidas.json\',receipts)\n    return paths\n\ndef box_group(obj,record,known=True):\n    x1,y1,x2,y2=map(float,obj[\'bbox\']);w,h=record[\'width\'],record[\'height\']\n    color=\'#b91c1c\' if obj[\'prioridade\']==\'alto\' else \'#a16207\' if obj[\'prioridade\']==\'medio\' else \'#526174\'\n    if not known:color=\'#2563eb\'\n    label=obj[\'classe\']+\' · \'+obj[\'prioridade\']\n    if \'confianca\' in obj:label+=f" · {obj[\'confianca\']:.2f}"\n    if not known:label+=\' · sem cobertura\'\n    font=max(12,w*.035);stroke=max(2,w*.003)\n    tx=max(0,min(x1,w-font));ty=max(font,min(y1,h-font*.2))\n    cls=\'\' if core.alert_on(obj) else \'nonalert\'\n    dash=f\' stroke-dasharray="{stroke*3:g} {stroke*2:g}"\' if not known else \'\'\n    return f\'<g class="box {cls}"><title>{html.escape(label)}</title><rect x="{x1:g}" y="{y1:g}" width="{max(0,x2-x1):g}" height="{max(0,y2-y1):g}" fill="none" stroke="{color}" stroke-width="{stroke:g}"{dash}/><text x="{tx:g}" y="{ty:g}" fill="{color}" stroke="white" stroke-width="{font*.09:g}" paint-order="stroke" font-size="{font:g}" font-family="sans-serif">{html.escape(label)}</text></g>\'\n\ndef gallery(selected,selection,paths,bundle,evaluations,policy,output):\n    output=Path(output);definitions=[];cards=[];summaries=[]\n    selection_by_id={r[\'id\']:r for r in selection}\n    image_rows={m:{c:{r[\'id\']:r for r in evaluations[m][c][\'images\']} for c in protocol.COHORTS} for m in protocol.MODELS}\n    for index,r in enumerate(selected,1):\n        if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel protegido na renderização.\')\n        file=Path(paths[r[\'id\']]);raw=file.read_bytes()\n        if hashlib.sha256(raw).hexdigest()!=r[\'sha256\']:raise ValueError(\'Imagem mudou antes de renderizar.\')\n        token=\'foto\'+str(index);mime=mimetypes.guess_type(file.name)[0] or \'image/jpeg\'\n        definitions.append(f\'<image id="{token}" href="data:{mime};base64,{base64.b64encode(raw).decode()}" width="{r["width"]}" height="{r["height"]}"/>\')\n        category=selection_by_id[r[\'id\']][\'categoria\'];panels=[]\n        gt=core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\']&(1<<g[\'classe_id\'])],r[\'width\'],r[\'height\'],policy)\n        views=[(\'Referência anotada\',gt,None)]\n        for model in protocol.MODELS:\n            preds=bundle[\'predictions\'][model][r[\'coorte\']][r[\'id\']]\n            views.append((\'Baseline\' if model==\'baseline\' else \'Candidato 10\',core.with_policy(preds,r[\'width\'],r[\'height\'],policy),model))\n        for title,objects,model in views:\n            boxes=\'\'.join(box_group(o,r,bool(r[\'known_mask\']&(1<<o[\'classe_id\']))) for o in objects)\n            svg=f\'<svg viewBox="0 0 {r["width"]} {r["height"]}" role="img" aria-label="{html.escape(title)} na imagem {index}"><use href="#{token}"/>{boxes}</svg>\'\n            if model:\n                row=image_rows[model][r[\'coorte\']][r[\'id\']]\n                messages=[]\n                for obj in objects:\n                    if core.alert_on(obj):\n                        message=policy.gerar_mensagem(obj,\'medio\')\n                        if not r[\'known_mask\']&(1<<obj[\'classe_id\']):message+=\' Classe sem cobertura de referência nesta imagem.\'\n                        messages.append(message)\n                message_html=\'<ul>\'+\'\'.join(\'<li>\'+html.escape(m)+\'</li>\' for m in messages)+\'</ul>\' if messages else \'<p>Nenhum alerta V2 neste limiar. Isso não atesta ausência de obstáculos.</p>\'\n                content=f\'<p class="counts">TP {row["tp_alerta"]} · FN {row["fn_alerta"]} · FP {row["fp_alerta"]} · omissões altas {row["alta_sem_alerta"]}</p>\'+message_html\n                summaries.append({\'id\':r[\'id\'],\'coorte\':r[\'coorte\'],\'modelo\':model,\'categoria\':category,\n                    \'mensagens\':messages,\'tp_alerta\':row[\'tp_alerta\'],\'fn_alerta\':row[\'fn_alerta\'],\'fp_alerta\':row[\'fp_alerta\'],\n                    \'alta_sem_alerta\':row[\'alta_sem_alerta\'],\'predicoes\':objects})\n            else:content=\'<p>Caixas anotadas nas classes com cobertura. Azul tracejado nas previsões indica classe sem cobertura; seu acerto/erro não é julgado.</p>\'\n            panels.append(\'<section class="panel"><h3>\'+title+\'</h3>\'+svg+content+\'</section>\')\n        cohort_label=\'Home VAL\' if r[\'coorte\']==protocol.COHORTS[0] else \'Open Images VAL\'\n        cards.append(f\'<article data-cohort="{r["coorte"]}"><h2>{index:02d} · {cohort_label} · {html.escape(CATEGORY_LABELS[category])}</h2><p class="id">{html.escape(r["id"])}</p><div class="panels">\'+\'\'.join(panels)+\'</div></article>\')\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Demonstração 19 — objetos e alertas V2</title><style>body{font:16px/1.55 system-ui;margin:0;background:#eef2f6;color:#162739}main{max-width:1550px;margin:auto;padding:24px}article,.intro{background:white;border:1px solid #d3dce5;border-radius:12px;padding:20px;margin:18px 0}.panels{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:20px}.panel{min-width:0}svg{width:100%;max-height:480px;background:#f1f3f5}.id{overflow-wrap:anywhere;color:#526174}.counts{font-weight:600}h1,h2,h3{line-height:1.25}h2{font-size:22px}h3{font-size:18px}.nonalert{display:none}.showall .nonalert{display:inline}button{padding:9px 14px;margin:4px;border:1px solid #8297ac;border-radius:6px;background:white;cursor:pointer}button:focus-visible{outline:3px solid #2563eb}li{margin:8px 0}@media(max-width:1000px){.panels{grid-template-columns:1fr}main{padding:10px}}[hidden]{display:none!important}</style><main><p>TCC · Demonstração em imagens de desenvolvimento</p><h1>Reconhecimento de objetos e mensagens da política V2</h1><div class="intro"><p>Mesma imagem, mesma confiança 0,25: referência anotada, baseline e candidato 10. As previsões foram capturadas anteriormente; esta página não executa inferência em novas imagens.</p><p>Galeria ilustrativa com acertos e erros escolhidos por regras e semente fixas. As métricas completas usam 456 Home VAL e 1.000 OI VAL; esta seleção não mede desempenho.</p><p>Vermelho: prioridade alta V2. Ocre: média. Azul tracejado: classe sem cobertura de anotação, sem julgamento de acerto/erro. A classificação usa posição e tamanho aparentes; não mede distância nem comprova risco físico.</p><p><label><input type="checkbox" onchange="document.body.classList.toggle(\'showall\',this.checked)"> Mostrar também objetos que não geram alerta V2</label></p><nav aria-label="Filtrar imagens"><button type="button" onclick="filtrar(\'todos\')">Todas</button><button type="button" onclick="filtrar(\'home_val_historica\')">Home</button><button type="button" onclick="filtrar(\'openimages_val_nova\')">Open Images</button></nav></div>\'\'\'\n    page+=\'<svg width="0" height="0" aria-hidden="true" style="position:absolute"><defs>\'+\'\'.join(definitions)+\'</defs></svg>\'\n    page+=\'\'.join(cards)+\'\'\'<p>Uso: demonstração acadêmica controlada. Reserva e testes não foram usados. Nenhum modelo foi substituído.</p></main><script>function filtrar(c){document.querySelectorAll(\'article\').forEach(a=>a.hidden=c!==\'todos\'&&a.dataset.cohort!==c)}</script></html>\'\'\'\n    path=output/\'demonstracao_19.html\';path.write_text(page,encoding=\'utf-8\')\n    core.save_json(output/\'demonstracao_predicoes_e_mensagens.json\',summaries)\n    return path\n', 'cons19_inputs.py': '"""Resultados autenticados e previsões já capturadas; sem Torch ou inferência."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv,gzip,hashlib,io,json,math,zipfile\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nimport conf14_evaluation as evaluator\nimport cons19_protocol as protocol\n\ndef rows(raw):return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\ndef checked_zip(path,sha):\n    if core.sha256_file(path)!=sha:raise ValueError(\'Pacote diferente do resultado analisado: \'+Path(path).name)\n    z=zipfile.ZipFile(path)\n    try:\n        if len(z.namelist())!=len(set(z.namelist())) or z.testzip():raise ValueError(\'Pacote com nomes repetidos ou CRC inválido.\')\n        for n in z.namelist():core.safe_member(n)\n        return z\n    except Exception:z.close();raise\n\ndef load(packages,expected,output):\n    output=Path(output);sources={};all_entries=[];references={};caches={}\n    for number in [16,17,18]:\n        with checked_zip(packages[number],expected[\'pacote_\'+str(number)]) as z:\n            member=\'resumo_calibracao_17.json\' if number==17 else f\'resumo_experimento_{number}.json\'\n            summary=json.loads(z.read(member))\n            if not summary[\'concluido\']:raise ValueError(\'Resultado anterior incompleto.\')\n            sources[number]={\'summary\':summary,\'sha256\':expected[\'pacote_\'+str(number)]}\n            if number==17:\n                if summary[\'eligible_calibration\']!=0 or summary[\'distinct_configurations_evaluated_verification\']!=0:\n                    raise ValueError(\'Resultado 17 diferente da decisão que encerrou a busca de limiares.\')\n                records=json.loads(gzip.decompress(z.read(\'registros_validacao.json.gz\')))\n                if len(records)!=1456 or len({r[\'id\'] for r in records})!=1456:raise ValueError(\'Registros de validação incompatíveis.\')\n                if Counter(r[\'role\'] for r in records)!={\'home_val\':456,\'val_nova\':1000}:raise ValueError(\'Papel protegido entrou nos registros.\')\n                sources[number][\'records\']=records\n            else:\n                if any(summary[k] for k in [\'reserva_futura_inferida\',\'test_home_inferido\',\'test_oi20k_inferido\',\'modelo_substituido\']):\n                    raise ValueError(\'Resultado anterior usou conjunto protegido ou substituiu modelo.\')\n                entries=json.loads(z.read(\'checkpoints_avaliados.json\'))\n                finals=[e for e in entries if e[\'arm\']!=\'inicial\' and e[\'epoch\']==3]\n                if len(finals)!=2 or summary[\'checkpoint_final_elegivel\'] is not None:raise ValueError(\'Finais anteriores incompatíveis.\')\n                for entry in entries:\n                    if entry[\'arm\']!=\'inicial\':all_entries.append({\'experiment\':number,**entry})\n                if number==18:\n                    cfg=json.loads(z.read(\'configuracao_experimento_18.json\'))\n                    if cfg[\'context_key\']!=expected[\'contexto_18\']:raise ValueError(\'Contexto 18 diferente.\')\n                    raw_rule=z.read(\'regra_selecao_congelada_12.json\')\n                    if hashlib.sha256(raw_rule).hexdigest()!=expected[\'regra_12_sha256\']:raise ValueError(\'Regra histórica diferente.\')\n                    (output/\'regra_selecao_congelada_12.json\').write_bytes(raw_rule)\n                    rule=json.loads(raw_rule)\n                    for model in protocol.MODELS:\n                        references[model]={};caches[model]={}\n                        for cohort in protocol.COHORTS:\n                            prefix=f\'referencias/{model}/{cohort}/\'\n                            meta=json.loads(z.read(prefix+\'cache/cache_manifesto.json\'))\n                            request=meta[\'solicitacao\'];weight=expected[\'baseline\' if model==\'baseline\' else \'candidato\']\n                            if not meta[\'concluido\'] or request[\'peso_sha256\']!=weight or request[\'coorte\']!=cohort:\n                                raise ValueError(\'Cache de outra referência.\')\n                            raw=z.read(prefix+\'predicoes_conf_025.jsonl.gz\')\n                            values=[json.loads(l) for l in gzip.decompress(raw).decode().splitlines() if l]\n                            references[model][cohort]={\'summary\':json.loads(z.read(prefix+\'metricas.json\')),\n                                \'images\':rows(z.read(prefix+\'alertas_por_imagem.csv\')),\n                                \'events\':data.parse_events(z.read(prefix+\'avaliacao_objetos.csv\')),\n                                \'classes\':rows(z.read(prefix+\'metricas_por_classe.csv\')),\'request\':request}\n                            caches[model][cohort]=values\n                            target=output/\'caches_reutilizados\'/model/(cohort+\'.jsonl.gz\')\n                            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)\n    records=sources[17][\'records\'];by_id={r[\'id\']:r for r in records};cohorts={};predictions={m:{} for m in protocol.MODELS}\n    for cohort in protocol.COHORTS:\n        ordered=[v[\'id\'] for v in caches[\'baseline\'][cohort]]\n        wanted={r[\'id\'] for r in records if r[\'coorte\']==cohort}\n        if len(ordered)!=len(wanted) or set(ordered)!=wanted:raise ValueError(\'Cache e GT usam IDs diferentes.\')\n        cohorts[cohort]=[by_id[iid] for iid in ordered]\n        for model in protocol.MODELS:\n            values=caches[model][cohort]\n            if [v[\'id\'] for v in values]!=ordered:raise ValueError(\'Ordem dos caches diverge entre modelos.\')\n            sig=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in cohorts[cohort]]\n            if hashlib.sha256(core.canonical(sig).encode()).hexdigest()!=references[model][cohort][\'request\'][\'entradas_sha256\']:\n                raise ValueError(\'Bytes, dimensões ou cobertura diferem da referência.\')\n            predictions[model][cohort]={}\n            for item in values:\n                r=by_id[item[\'id\']]\n                if item[\'image_sha256\']!=r[\'sha256\'] or item[\'label_sha256\']!=r[\'label_sha256\']:raise ValueError(\'Cache de outros bytes.\')\n                evaluator.validate_predictions(item[\'predicoes\'],r)\n                if any(p[\'confianca\']<.25 for p in item[\'predicoes\']):raise ValueError(\'Cache compacto abaixo do limiar congelado.\')\n                predictions[model][cohort][r[\'id\']]=item[\'predicoes\']\n    core.save_json(output/\'fontes_autenticadas_19.json\',{str(n):{\'sha256\':s[\'sha256\'],\'status\':s[\'summary\'][\'status\']} for n,s in sources.items()})\n    with gzip.open(output/\'dados_validacao_reutilizados.json.gz\',\'wt\',encoding=\'utf-8\') as f:json.dump(records,f,ensure_ascii=False)\n    return {\'sources\':sources,\'entries\':all_entries,\'references\':references,\'cohorts\':cohorts,\'predictions\':predictions,\'rule\':rule}\n\ndef reproduce(bundle,policy,matcher,output):\n    evaluations={m:{} for m in protocol.MODELS};checks=[]\n    integer_fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\n                    \'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf\']\n    for model in protocol.MODELS:\n        for cohort,records in bundle[\'cohorts\'].items():\n            result=evaluator.evaluate_cache(records,bundle[\'predictions\'][model][cohort],.25,policy,matcher,Path(output)/\'avaliacoes_reutilizadas\'/model/cohort)\n            original=bundle[\'references\'][model][cohort];old_images={r[\'id\']:r for r in original[\'images\']}\n            image_checks=0\n            for row in result[\'images\']:\n                for k in integer_fields:\n                    if row[k]!=int(old_images[row[\'id\']][k]):raise ValueError(\'Contador por imagem divergente: \'+model+\'/\'+cohort+\'/\'+row[\'id\']+\'/\'+k)\n                    image_checks+=1\n            oldgt=data.gt_map(original[\'events\']);newgt=data.gt_map(evaluator.parse_current_events(result[\'events\']))\n            if oldgt.keys()!=newgt.keys():raise ValueError(\'Referências GT divergentes.\')\n            for key,a in oldgt.items():\n                b=newgt[key]\n                if any(a[k]!=b[k] for k in [\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']):\n                    raise ValueError(\'Estado de objeto divergente na reprodução.\')\n            for k,v in original[\'summary\'][\'alertas_por_objeto\'].items():\n                if result[\'summary\'][\'alertas_por_objeto\'][k]!=v:raise ValueError(\'Métrica de alerta divergente.\')\n            evaluations[model][cohort]=result\n            checks.append({\'modelo\':model,\'coorte\':cohort,\'checagens_imagem\':image_checks,\'objetos_GT\':len(oldgt),\'reproduziu\':True})\n            print(\'Previsões reaproveitadas e referência conferida:\',model,cohort,flush=True)\n    core.save_json(Path(output)/\'reproducao_referencias_19.json\',checks)\n    return evaluations,checks\n', 'cons19_io.py': '"""Drive, recibos atômicos e pacote leve; credenciais ficam fora dos resultados."""\nfrom __future__ import annotations\nimport base64\nimport configparser\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport zipfile\n\nimport diag11_core as core\n\nclass DigestMismatch(RuntimeError):\n    pass\n\nclass Drive:\n    def __init__(self, binary, config, root=\'gdrive_tcc:TCC_Visao_Computacional\'):\n        self.binary, self.config, self.root = str(binary), str(config), root\n\n    @classmethod\n    def from_kaggle(cls):\n        from kaggle_secrets import UserSecretsClient\n        root = Path(\'/tmp/tcc_rclone\')\n        root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        root.chmod(0o700)\n        try:\n            secret = UserSecretsClient().get_secret(\'GDRIVE_RCLONE_CONFIG_B64\')\n        except Exception:\n            raise RuntimeError(\'Habilite o Secret existente GDRIVE_RCLONE_CONFIG_B64 em Add-ons → Secrets.\') from None\n        try:\n            raw = base64.b64decode(secret.strip(), validate=True)\n            cfg = configparser.ConfigParser(interpolation=None)\n            cfg.read_string(raw.decode(\'utf-8-sig\'))\n        except Exception:\n            raise RuntimeError(\'A configuração do Secret rclone não é válida.\') from None\n        if (not cfg.has_section(\'gdrive_tcc\') or cfg.get(\'gdrive_tcc\', \'type\', fallback=\'\') != \'drive\'\n                or not cfg.get(\'gdrive_tcc\', \'client_id\', fallback=\'\')\n                or not cfg.get(\'gdrive_tcc\', \'token\', fallback=\'\')\n                or cfg.get(\'gdrive_tcc\', \'scope\', fallback=\'\') != \'drive\'):\n            raise RuntimeError(\'O Secret precisa da conexão gdrive_tcc autorizada para leitura e gravação em Meu Drive.\')\n        cfg.remove_option(\'gdrive_tcc\', \'team_drive\')\n        config = root / \'gdrive_tcc.conf\'\n        with config.open(\'w\', encoding=\'utf-8\') as f:\n            cfg.write(f)\n        config.chmod(0o600)\n        del secret, raw, cfg\n        binary = root / \'rclone\'\n        if not binary.is_file():\n            executable = shutil.which(\'rclone\')\n            if not executable:\n                import os\n                prefix = [] if os.geteuid() == 0 else [\'sudo\']\n                subprocess.run(prefix + [\'apt-get\', \'update\', \'-qq\'], check=True)\n                subprocess.run(prefix + [\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'rclone\'], check=True)\n                executable = shutil.which(\'rclone\')\n            if not executable:\n                raise RuntimeError(\'Executável rclone indisponível após a instalação.\')\n            shutil.copy2(executable, binary)\n            binary.chmod(0o700)\n        return cls(binary, config)\n\n    def _call(self, args):\n        return subprocess.run([self.binary, \'--config\', self.config] + list(args)\n                              + [\'--retries\', \'2\', \'--low-level-retries\', \'3\',\n                                 \'--contimeout\', \'15s\', \'--timeout\', \'60s\'],\n                              capture_output=True, text=True)\n\n    def exists(self, relative):\n        result = self._call([\'lsjson\', self.root + \'/\' + relative, \'--stat\'])\n        if result.returncode == 0:\n            return True\n        if result.returncode in {3, 4}:\n            return False\n        raise RuntimeError(\'Falha ao consultar Drive (código \' + str(result.returncode) + \'): \' + relative)\n\n    def fetch(self, relative, target, expected_sha=None, optional=False):\n        target = Path(target)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        receipt = target.with_name(target.name + \'.download.json\')\n        if target.is_file() and target.stat().st_size:\n            local_sha = core.sha256_file(target)\n            prior = {}\n            if receipt.is_file():\n                try:\n                    prior = json.loads(receipt.read_text())\n                except (ValueError, OSError):\n                    pass\n            if ((expected_sha and local_sha == expected_sha)\n                    or (not expected_sha and prior.get(\'origem\') == relative and prior.get(\'sha256\') == local_sha)):\n                print(\'Entrada local conferida:\', relative, flush=True)\n                return target\n        if optional and not self.exists(relative):\n            return None\n        print(\'Baixando:\', relative, flush=True)\n        tmp = target.with_name(target.name + \'.part\')\n        result = self._call([\'copyto\', self.root + \'/\' + relative, str(tmp)])\n        if result.returncode:\n            raise RuntimeError(\'Download interrompido (código \' + str(result.returncode) + \'): \' + relative)\n        if not tmp.is_file() or not tmp.stat().st_size:\n            raise RuntimeError(\'Download vazio: \' + relative)\n        actual = core.sha256_file(tmp)\n        if expected_sha and actual != expected_sha:\n            tmp.replace(target.with_name(target.name + \'.rejeitado_\' + actual[:12]))\n            raise DigestMismatch(\'SHA-256 diferente da fonte autenticada: \' + relative)\n        tmp.replace(target)\n        core.save_json(receipt, {\'origem\': relative, \'sha256\': actual, \'bytes\': target.stat().st_size})\n        return target\n\n    def backup(self, local, relative, required=True):\n        print(\'Salvando resultados no Drive...\', flush=True)\n        result = self._call([\'copy\', str(local), self.root + \'/\' + relative,\n                             \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\', \'--exclude\', \'**/__pycache__/**\'])\n        if result.returncode and required:\n            raise RuntimeError(\'Falha no salvamento no Drive; resultados locais preservados. Código \' + str(result.returncode))\n        return result.returncode == 0\n\n    def restore(self, relative, local):\n        if not self.exists(relative):\n            return False\n        result = self._call([\'copy\', self.root + \'/\' + relative, str(local),\n                             \'--exclude\', \'*.zip\', \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\'])\n        if result.returncode:\n            raise RuntimeError(\'Não foi possível recuperar o resultado anterior; código \' + str(result.returncode))\n        return True\n\ndef package(directory):\n    directory=Path(directory);target=directory/\'pacote_consolidacao_19.zip\';tmp=target.with_suffix(\'.tmp\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for path in sorted(directory.rglob(\'*\')):\n            if not path.is_file() or path in {target,tmp} or path.suffix in {\'.pt\',\'.tmp\',\'.part\'}:continue\n            rel=path.relative_to(directory)\n            if \'__pycache__\' in rel.parts:continue\n            z.write(path,rel.as_posix())\n    tmp.replace(target)\n    return target\n\ndef failure(directory,error,stage,drive=None,remote=None):\n    core.save_json(Path(directory)/\'execucao_interrompida_19.json\',{\'concluido\':False,\'etapa\':stage,\n        \'tipo\':type(error).__name__,\'mensagem\':str(error)[:1400],\'treinamento_executado\':False,\n        \'inferencia_executada\':False,\'reserva_inferida\':False,\'modelo_substituido\':False})\n    result=package(directory)\n    if drive and remote:\n        try:drive.backup(directory,remote,required=False)\n        except Exception:pass\n    return result\n', 'cons19_protocol.py': '"""Consolidação e protocolo da avaliação posterior, sem abrir a reserva."""\nfrom __future__ import annotations\nimport hashlib,json\n\nVERSION=\'19-consolidacao-demonstracao-1.0.0\'\nMODELS=(\'baseline\',\'candidato_10\')\nCOHORTS=(\'home_val_historica\',\'openimages_val_nova\')\nSPEC={\'version\':VERSION,\'models\':list(MODELS),\'confidence\':.25,\'policy\':\'v2\',\'iou_match\':.5,\n      \'training\':False,\'inference\':False,\'demo_images_per_cohort\':12,\'demo_per_category\':2,\n      \'demo_seed\':\'19-galeria-42\',\'demo_categories\':[\'mais_tp_candidato10\',\'mais_tp_baseline\',\n          \'omissao_alta\',\'falso_alerta\',\'acerto_sem_erro\',\'sem_alerta_referencia\'],\n      \'selection\':\'categorias ilustrativas com ordem por SHA; sem trocar pesos/limiares\',\n      \'evaluation\':\'duas validacoes ja expostas; todas as imagens, sem usar a galeria como metrica\',\n      \'final_reserve_opened\':False,\'new_weight_selection\':False}\ndef canonical(x):return json.dumps(x,sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False)\ndef digest(x):return hashlib.sha256(canonical(x).encode()).hexdigest()\n\ndef final_protocol(expected):\n    """Contagens absolutas das 1.000 VAL não viram limites artificiais nas 300."""\n    return {\'version\':\'20-plano-avaliacao-final-congelado-em-19-v1\',\'status\':\'planejado; reserva nao consultada no 19\',\n        \'primary_model\':\'baseline\',\'comparison_model\':\'candidato_10\',\n        \'weight_sha256\':{\'baseline\':expected[\'baseline\'],\'candidato_10\':expected[\'candidato\']},\n        \'dataset_sha256\':expected[\'dataset_novo\'],\'archive_sha256\':expected[\'zip_novo\'],\n        \'dataset_location\':expected[\'pasta_validacao\']+\'/\'+expected[\'arquivo_validacao\'],\n        \'role\':\'reserva_futura\',\'images\':300,\'population\':\'todos os 300 IDs do papel reserva_futura no manifesto autenticado; sem substituicoes\',\n        \'classes\':\'mesmo mapeamento de 12 classes; cobertura parcial original preservada\',\n        \'policy_sha256\':expected[\'politica\'],\'matcher_sha256\':expected[\'associador\'],\n        \'inference\':{\'ultralytics\':\'8.4.150\',\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.7,\'max_det\':300,\'half\':False,\'batch\':1},\n        \'alert_confidence\':.25,\'match_iou\':.5,\'policy\':\'v2\',\n        \'primary_estimands\':[\'recall de alertas por objeto do baseline\',\'delta recall candidato10 menos baseline\'],\n        \'secondary\':[\'precisao\',\'F1\',\'omissoes altas: contagem e taxa por objetos GT de prioridade alta\',\n            \'metricas por classe e suporte\',\'contagens TP/FN/FP e denominadores\'],\n        \'confidence_intervals\':{\'method\':\'bootstrap pareado de grupos de imagens; manter bytes iguais e pHash64 distancia<=4 no mesmo grupo por ligacao transitiva\',\n            \'repetitions\':2000,\'seed\':19002026,\'confidence\':.95,\'interval\':\'percentil 2.5 e 97.5; grupos reamostrados com reposicao; mesmos grupos para ambos os modelos\',\n            \'zero_denominator\':\'replica sem suporte fica indefinida; informar quantidade de replicas validas por metrica\'},\n        \'low_support\':\'informar contagens; sinalizar classe com menos de 30 GT, sem ocultar resultados\',\n        \'weight_or_threshold_tuning_after_reserve\':False,\'second_choice_after_failure\':False,\n        \'deployment_decision\':\'nenhuma troca automatica; caracterizacao dos dois modelos predefinidos\',\n        \'historical_rules\':\'os seis limites das validacoes ficam no desenvolvimento; nao aplicar limites absolutos de 1.000 imagens em 300\',\n        \'home_test\':\'fora desta avaliacao primaria; historico original do baseline nao comprovado\',\n        \'openimages_test20k\':\'fora desta etapa\',\'training\':False,\n        \'limits\':[\'A independencia refere-se a reserva preservada no fluxo documentado; nao comprova ausencia de exposicao em todo pretreino original.\',\n                  \'Nao valida distancia, risco fisico ou uso assistivo em ambiente real.\']}\n', 'cons19_reports.py': '"""Tabelas consolidadas, texto para o TCC e pacote da demonstração."""\nfrom __future__ import annotations\nimport html,json\nfrom pathlib import Path\nimport diag11_core as core\nfrom exp12_protocol import judge\nimport cons19_protocol as protocol\n\ndef table(rows):\n    if not rows:return \'<p>Sem registros.</p>\'\n    keys=list(rows[0]);return \'<div class="scroll"><table><thead><tr>\'+\'\'.join(\'<th>\'+html.escape(str(k))+\'</th>\' for k in keys)+\'</tr></thead><tbody>\'+\'\'.join(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(r.get(k,\'\')))+\'</td>\' for k in keys)+\'</tr>\' for r in rows)+\'</tbody></table></div>\'\ndef percentage(v):return \'sem suporte\' if v is None else f\'{100*v:.2f}%\'.replace(\'.\',\',\')\n\ndef consolidate(bundle,output):\n    output=Path(output);rows=[]\n    entries=[]\n    for model in protocol.MODELS:\n        refs=bundle[\'references\'][model]\n        entries.append({\'experiment\':\'referencia\',\'id\':model,\'epoch\':0,\'updates\':0,\n            \'home\':refs[protocol.COHORTS[0]][\'summary\'],\'external\':refs[protocol.COHORTS[1]][\'summary\'],\'reference\':True})\n    entries += [{**e,\'reference\':False} for e in bundle[\'entries\']]\n    for e in entries:\n        checks=None if e[\'reference\'] else judge(e[\'home\'],e[\'external\'],bundle[\'rule\'])\n        row={\'experimento\':e[\'experiment\'],\'modelo\':e[\'id\'],\'epoca\':e[\'epoch\'],\'atualizacoes\':e[\'updates\'],\n            \'papel\':\'referencia fixa\' if e[\'reference\'] else \'final predefinido\' if e[\'epoch\']==3 else \'diagnostico intermediario\'}\n        for side,prefix in [(\'home\',\'home\'),(\'external\',\'externa\')]:\n            s=e[side];row.update({prefix+\'_\'+k:v for k,v in s[\'alertas_por_objeto\'].items()})\n            row[prefix+\'_alta_sem_alerta\']=s[\'alta_sem_alerta\'];row[prefix+\'_alta_ref\']=s[\'alta_ref\']\n        row[\'criterios_atendidos\']=None if checks is None else sum(v for k,v in checks.items() if k not in {\'elegivel\',\'retencao_externa\'})\n        row[\'elegivel_pelos_seis\']=None if checks is None else checks[\'elegivel\']\n        rows.append(row)\n    core.save_csv(output/\'comparacao_todos_checkpoints.csv\',rows)\n    finals=[r for r in rows if r[\'papel\']!=\'diagnostico intermediario\']\n    core.save_csv(output/\'comparacao_finais_e_referencias.csv\',finals)\n    experiments=[{\'experimento\':16,\'hipotese\':\'Wardrobe desconhecida em Home durante o ajuste\',\n        \'desfecho\':\'Nenhum final elegível; efeito pequeno entre os dois braços\'},\n        {\'experimento\':17,\'hipotese\':\'Limiares por classe para cadeira e mesa\',\'desfecho\':\'0/49 perfis elegíveis na calibração; nenhuma nova escolha na verificação\'},\n        {\'experimento\':18,\'hipotese\':\'Aumentar OI por batch de 4 para 8\',\'desfecho\':\'Pequeno ganho Home; retenção externa não preservada; 2/6 critérios no final 8/8\'}]\n    core.save_csv(output/\'resumo_hipoteses_16_18.csv\',experiments)\n    return rows,finals,experiments\n\ndef finish(bundle,evaluations,checks,selection,final_plan,context,output):\n    output=Path(output);rows,finals,experiments=consolidate(bundle,output)\n    shown=[]\n    for r in finals:\n        shown.append({\'Modelo\':r[\'modelo\'],\'P Home\':percentage(r[\'home_precision\']),\'R Home\':percentage(r[\'home_recall\']),\n            \'Altas omitidas Home\':r[\'home_alta_sem_alerta\'],\'P externa\':percentage(r[\'externa_precision\']),\n            \'R externa\':percentage(r[\'externa_recall\']),\'Altas omitidas externas\':r[\'externa_alta_sem_alerta\']})\n    gt_counts={c:len(v) for c,v in bundle[\'cohorts\'].items()}\n    summary={\'version\':protocol.VERSION,\'concluido\':True,\'status\':\'consolidacao_e_demonstracao_concluidas\',\n        \'context_key\':context,\'modelo_principal\':\'baseline\',\'comparador\':\'candidato_10\',\'novos_pesos\':False,\n        \'treinamento_executado\':False,\'inferencia_executada\':False,\'previsoes_reutilizadas\':True,\n        \'validacoes\':gt_counts,\'imagens_demonstracao\':len(selection),\'galeria_ilustrativa\':True,\n        \'selecao_demonstracao_sha256\':protocol.digest(selection),\'protocolo_final_sha256\':protocol.digest(final_plan),\n        \'reserva_futura_extraida\':False,\'reserva_futura_inferida\':False,\'test_home_inferido\':False,\'test_oi20k_inferido\':False,\n        \'modelo_substituido\':False,\'preenchimento_manual_obrigatorio\':False,\'referencias_reproduzidas\':all(r[\'reproduziu\'] for r in checks),\n        \'decisao\':\'Encerrar os ajustes curtos desta linha; consolidar resultados e demonstrar o comportamento dos dois modelos fixos.\',\n        \'limites\':[\'As validações já influenciaram o desenvolvimento; não são teste final independente.\',\n            \'A galeria não mede desempenho e inclui exemplos de acertos e falhas por regras fixas.\',\n            \'Cobertura parcial OI: classe desconhecida não vira falso positivo nem negativo conhecido.\',\n            \'V2 usa geometria aparente e classe, sem medir distância ou risco físico validado.\',\n            \'Origem do treinamento original do baseline permanece não determinada.\']}\n    core.save_json(output/\'resumo_consolidacao_19.json\',summary)\n    text=\'\'\'# Consolidação dos experimentos e demonstração do sistema\n\nO desenvolvimento avaliou um detector inicial, denominado baseline, e o candidato 10, ajustado com imagens Home e Open Images. A avaliação de alertas usou a política V2, confiança de 0,25 e associação por classe com IoU mínimo de 0,5. As métricas de alerta representam concordância com essa política, sem validação independente de risco físico.\n\nForam mantidas duas validações de desenvolvimento: 456 imagens Home e 1.000 imagens externas. A avaliação externa respeitou a cobertura parcial das anotações: classes sem cobertura não foram presumidas como negativas. Esses conjuntos já influenciaram o desenvolvimento e não constituem teste final independente.\n\n\'\'\'\n    text+=\'| Modelo | Precisão Home | Recall Home | Omissões altas Home | Precisão externa | Recall externo | Omissões altas externas |\\n| --- | ---: | ---: | ---: | ---: | ---: | ---: |\\n\'\n    for r in finals:text+=\'| \'+r[\'modelo\']+\' | \'+\' | \'.join([percentage(r[\'home_precision\']),percentage(r[\'home_recall\']),str(r[\'home_alta_sem_alerta\']),percentage(r[\'externa_precision\']),percentage(r[\'externa_recall\']),str(r[\'externa_alta_sem_alerta\'])])+\' |\\n\'\n    text+=\'\'\'\nNo experimento 16, o ajuste da supervisão de armários em Home apresentou efeito pequeno e nenhum checkpoint final cumpriu os seis requisitos históricos. No 17, nenhuma das 49 combinações de limiares de cadeira e mesa foi elegível na calibração; por esse motivo, nenhuma nova configuração foi submetida à verificação interna. No 18, o batch equilibrado 8/8 apresentou pequeno ganho em Home sobre 12/4, sem preservar o recall e as omissões altas na validação externa. Aumentar a exposição a Open Images nesse orçamento curto não produziu a retenção externa pretendida.\n\nO baseline foi mantido como referência principal, e o candidato 10 como comparador relevante para o domínio externo. Essa decisão não afirma superioridade universal do baseline. Os demais modelos continuam documentados, sem escolher retrospectivamente uma época intermediária para substituir os desfechos finais.\n\nFoi preparada uma demonstração com as mesmas imagens para referência anotada, baseline e candidato 10. A seleção usa categorias fixas de acertos, omissões, falsos alertas e diferenças entre os modelos, com ordenação determinística por hash. A galeria é ilustrativa; as métricas da tabela usam todas as imagens das duas validações. As mensagens consideram classe, posição e tamanho aparente segundo a V2; não medem distância real.\n\nA reserva externa de 300 imagens permanece sem extração ou inferência nesta consolidação. O protocolo de avaliação posterior fixa os dois pesos, a política e os limiares antes de sua consulta. Prevê métricas e denominadores completos, comparação pareada e intervalos de confiança por grupos de imagens. Os limites absolutos de omissões da validação de 1.000 imagens não serão transportados para a reserva de 300. A consulta futura não autoriza ajustes retrospectivos para melhorar o mesmo resultado.\n\nAs limitações incluem uma semente e orçamento curto nos ajustes, diferenças de domínio e de anotação, cobertura parcial externa e histórico original do treino do baseline não comprovado. Os experimentos não demonstram que o conjunto completo de 100 mil imagens seja inadequado, pois o ajuste 18 usou somente o replay de 2 mil imagens Open Images. A demonstração é acadêmica e controlada, sem comprovação de segurança assistiva.\n\'\'\'\n    (output/\'texto_base_capitulo4.md\').write_text(text,encoding=\'utf-8\')\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Consolidação 19 — resultados e demonstração</title><style>body{font:16px/1.6 system-ui;background:#eef2f6;color:#162739;margin:0}main{max-width:1450px;margin:auto;padding:28px}.note{background:white;border:1px solid #d3dce5;border-radius:12px;padding:20px;margin:20px 0}.scroll{overflow:auto}table{border-collapse:collapse;background:white;width:100%}td,th{padding:10px;border:1px solid #d3dce5;text-align:left}h1,h2{line-height:1.2}a{color:#1d4ed8}</style><main><p>TCC · Consolidação experimental</p><h1>Resultados, demonstração e avaliação posterior</h1><div class="note"><strong>Baseline como referência principal; candidato 10 como comparador.</strong><p>Os ajustes 16–18 não produziram um novo modelo elegível nos seis critérios congelados. Esta etapa reaproveita previsões e não treina ou consulta testes.</p><p><a href="demonstracao_19.html">Abrir a demonstração comparativa</a> · <a href="texto_base_capitulo4.md">Texto-base para o capítulo experimental</a></p></div><h2>Finais e referências nas duas validações</h2>\'\'\'\n    page+=table(shown)+\'<h2>Hipóteses encerradas nesta linha</h2>\'+table(experiments)\n    page+=\'<h2>Avaliação final planejada</h2><p>300 imagens externas do papel reserva_futura, sem consulta nesta etapa. Pesos, limiares, política, associação e análise estatística foram fixados em protocolo_avaliacao_final.json. Baseline e candidato 10 serão avaliados, sem escolha adaptativa ou troca automática de modelo.</p>\'\n    page+=\'<h2>Limites</h2><ul>\'+\'\'.join(\'<li>\'+html.escape(v)+\'</li>\' for v in summary[\'limites\'])+\'</ul></main></html>\'\n    (output/\'relatorio_consolidacao_19.html\').write_text(page,encoding=\'utf-8\')\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Notebook 19 concluído\\n\\nAbra demonstracao_19.html para comparar os dois modelos nas imagens. Abra relatorio_consolidacao_19.html para as tabelas. O texto-base está em texto_base_capitulo4.md. A avaliação final está planejada em protocolo_avaliacao_final.json, sem abrir a reserva.\\n\\nResultados em pacote_consolidacao_19.zip. Nenhum novo treino, inferência ou ajuste manual foi executado.\\n\',encoding=\'utf-8\')\n    return summary\n', 'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'src/__init__.py': '', 'src/politica_alertas.py': '"""Parâmetros e funções da V2 original, sem reestimar com o TEST."""\nimport math\n\nNOMES_PT = {"bed": "cama", "sofa": "sofá", "chair": "cadeira", "table": "mesa",\n            "wardrobe": "guarda-roupa", "potted plant": "planta em vaso"}\nVERBOS_PT = {"bed": "detectada", "sofa": "detectado", "chair": "detectada", "table": "detectada",\n             "wardrobe": "detectado", "potted plant": "detectada"}\nORDEM_PRIORIDADE = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef avaliar_bbox(classe, bbox, largura, altura, versao="v2"):\n    if versao not in {"v1", "v2"}:\n        raise ValueError("Versão deve ser v1 ou v2.")\n    x1, y1, x2, y2 = map(float, bbox)\n    if largura <= 0 or altura <= 0 or not all(math.isfinite(v) for v in [x1, y1, x2, y2]):\n        raise ValueError("Dimensões ou coordenadas inválidas.")\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("Caixa sem área positiva.")\n    posicao = classificar_posicao((x1 + x2) / 2, largura)\n    tamanho, area = classificar_tamanho((x2 - x1) * (y2 - y1), largura * altura)\n    vertical, base = classificar_posicao_vertical(y2, altura)\n    resultado = (classificar_risco_v2(classe, posicao, tamanho, vertical) if versao == "v2"\n                 else classificar_risco_v1(classe, posicao, tamanho))\n    nivel = resultado["nivel_risco"]\n    return {"prioridade": nivel if nivel in {"baixo", "medio", "alto"} else "sem_prioridade",\n            "status_original": nivel, "pontuacao": resultado["pontuacao"],\n            "posicao": posicao, "tamanho": tamanho, "regiao_vertical": vertical,\n            "area_relativa": area, "base_vertical_relativa": base,\n            "ajuste_vertical": resultado.get("ajuste_vertical", 0)}\n\n\ndef gerar_mensagem(objeto, nivel_minimo="medio"):\n    if nivel_minimo not in {"medio", "alto"}:\n        raise ValueError("Nível mínimo deve ser medio ou alto.")\n    prioridade = objeto["prioridade"]\n    if ORDEM_PRIORIDADE.get(prioridade, 0) < ORDEM_PRIORIDADE[nivel_minimo]:\n        return ""\n    posicao = {"centro": "central", "esquerda": "esquerda", "direita": "direita"}[objeto["posicao"]]\n    classe = objeto["classe"]\n    return (f"Atenção: {NOMES_PT[classe]} {VERBOS_PT[classe]} na região {posicao} da imagem. "\n            f"Prioridade {\'alta\' if prioridade == \'alto\' else \'média\'}.")\n\n\ndef gerar_mensagens_tabela(df, coluna="nivel_risco_v2"):\n    if df.empty:\n        return []\n    candidatos = df.loc[df[coluna].isin(["medio", "alto"])].copy()\n    candidatos["_ordem"] = candidatos[coluna].map(ORDEM_PRIORIDADE)\n    candidatos = candidatos.sort_values(["_ordem", "ocupacao_imagem"], ascending=[False, False])\n    return [gerar_mensagem({"classe": r["classe"], "posicao": r["posicao"], "prioridade": r[coluna]})\n            for _, r in candidatos.iterrows()]\n\nPESO_OBJETO = {\n    "bed": 3,\n    "sofa": 3,\n    "chair": 3,\n    "table": 3,\n    "wardrobe": 3,\n    "potted plant": 2,\n}\n\nCLASSES_CONTEXTUAIS = {\n    "lamp",\n    "door",\n}\n\nCLASSES_IGNORADAS = {\n    "tv",\n    "laptop",\n    "window",\n    "photo frame",\n}\n\n\nLIMITE_ESQUERDA = 1 / 3\nLIMITE_DIREITA = 2 / 3\n\nLIMITE_AREA_MEDIA = 0.05\nLIMITE_AREA_GRANDE = 0.20\n\nLIMITE_VERTICAL_SUPERIOR = 0.60\nLIMITE_VERTICAL_INFERIOR = 0.80\n\nPESO_POSICAO = {\n    "esquerda": 1,\n    "centro": 3,\n    "direita": 1,\n}\n\nPESO_TAMANHO = {\n    "pequeno": 1,\n    "medio": 2,\n    "grande": 3,\n}\n\nAJUSTE_VERTICAL = {\n    "superior": -2,\n    "intermediaria": -1,\n    "inferior": 0,\n}\n\ndef classificar_posicao(\n    centro_x: float,\n    largura_imagem: int,\n) -> str:\n    centro_normalizado = centro_x / largura_imagem\n\n    if centro_normalizado < LIMITE_ESQUERDA:\n        return "esquerda"\n\n    if centro_normalizado < LIMITE_DIREITA:\n        return "centro"\n\n    return "direita"\n\n\ndef classificar_tamanho(\n    area_box: float,\n    area_imagem: float,\n):\n    proporcao = area_box / area_imagem\n\n    if proporcao < LIMITE_AREA_MEDIA:\n        return "pequeno", proporcao\n\n    if proporcao < LIMITE_AREA_GRANDE:\n        return "medio", proporcao\n\n    return "grande", proporcao\n\n\ndef classificar_posicao_vertical(\n    y2: float,\n    altura_imagem: int,\n):\n    proporcao_vertical = y2 / altura_imagem\n\n    if proporcao_vertical < LIMITE_VERTICAL_SUPERIOR:\n        return "superior", proporcao_vertical\n\n    if proporcao_vertical < LIMITE_VERTICAL_INFERIOR:\n        return "intermediaria", proporcao_vertical\n\n    return "inferior", proporcao_vertical\n\ndef classificar_risco_v1(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n):\n    if classe in CLASSES_IGNORADAS:\n        return {\n            "pontuacao": 0,\n            "nivel_risco": "ignorado",\n            "motivo": "classe sem relevância direta para locomoção",\n        }\n\n    if classe in CLASSES_CONTEXTUAIS:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "contextual",\n            "motivo": "classe exige informação adicional de contexto",\n        }\n\n    if classe not in PESO_OBJETO:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "nao_mapeado",\n            "motivo": "classe não configurada",\n        }\n\n    pontuacao = (\n        PESO_OBJETO[classe]\n        + PESO_POSICAO[posicao]\n        + PESO_TAMANHO[tamanho]\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "motivo": (\n            f"{classe} + posição {posicao} + tamanho {tamanho}"\n        ),\n    }\n\ndef classificar_risco_v2(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n    posicao_vertical: str,\n):\n    avaliacao_v1 = classificar_risco_v1(\n        classe=classe,\n        posicao=posicao,\n        tamanho=tamanho,\n    )\n\n    if avaliacao_v1["nivel_risco"] in {\n        "ignorado",\n        "contextual",\n        "nao_mapeado",\n    }:\n        return {\n            "pontuacao": avaliacao_v1["pontuacao"],\n            "nivel_risco": avaliacao_v1["nivel_risco"],\n            "ajuste_vertical": 0,\n            "motivo": avaliacao_v1["motivo"],\n        }\n\n    ajuste_vertical = AJUSTE_VERTICAL[posicao_vertical]\n\n    pontuacao = max(\n        1,\n        avaliacao_v1["pontuacao"] + ajuste_vertical,\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "ajuste_vertical": ajuste_vertical,\n        "motivo": (\n            f"{classe} + posição {posicao} "\n            f"+ tamanho {tamanho} "\n            f"+ região vertical {posicao_vertical}"\n        ),\n    }\n', 'src/validacao_alertas.py': '"""Compara detectores na validação Home, com a V2 e os parâmetros do 05d."""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .politica_alertas import avaliar_bbox\n\nIMGSZ = 640\nCONF_ALERTA = 0.25\nIOU_NMS = 0.70\nIOU_CORRESPONDENCIA = 0.50\nMAX_DETECCOES = 300\n\n\ndef ler_objetos_yolo(caminho, largura, altura, nomes):\n    objetos = []\n    for n, linha in enumerate(Path(caminho).read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not linha.strip():\n            continue\n        partes = linha.split()\n        if len(partes) != 5:\n            raise ValueError(f"Label inválido: {caminho}, linha {n}")\n        valores = [float(p) for p in partes]\n        if not all(math.isfinite(v) for v in valores):\n            raise ValueError(f"Label não finito: {caminho}, linha {n}")\n        id_float, xc, yc, w, h = valores\n        c = int(id_float)\n        bordas = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]\n        if id_float != c or c not in nomes or w <= 0 or h <= 0 or min(bordas) < -0.0001 or max(bordas) > 1.0001:\n            raise ValueError(f"Classe ou caixa inválida: {caminho}, linha {n}")\n        x1, y1, x2, y2 = np.clip(bordas, 0, 1)\n        objetos.append({"classe_id": c, "classe": nomes[c],\n                        "bbox": [float(x1 * largura), float(y1 * altura),\n                                 float(x2 * largura), float(y2 * altura)]})\n    return objetos\n\n\ndef carregar_home_val(dataset, nomes):\n    dataset = Path(dataset)\n    raiz_imagens = dataset / "images/val"\n    registros = []\n    for imagem in sorted(raiz_imagens.rglob("*")):\n        if not imagem.is_file() or imagem.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:\n            continue\n        relativo = imagem.relative_to(raiz_imagens)\n        if relativo.parts[0] == "sunrgbd":\n            continue\n        label = dataset / "labels/val" / relativo.with_suffix(".txt")\n        with Image.open(imagem) as im:\n            largura, altura = im.size\n            im.verify()\n        registros.append({"imagem": imagem, "nome": relativo.as_posix(),\n                          "largura": largura, "altura": altura,\n                          "gt": ler_objetos_yolo(label, largura, altura, nomes)})\n    if not registros:\n        raise ValueError("Validação Home vazia.")\n    return registros\n\n\ndef iou_matriz(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)), dtype=float)\n    g = np.asarray([o["bbox"] for o in gt], dtype=float)\n    p = np.asarray([o["bbox"] for o in pred], dtype=float)\n    inicio = np.maximum(g[:, None, :2], p[None, :, :2])\n    fim = np.minimum(g[:, None, 2:], p[None, :, 2:])\n    inter = np.prod(np.maximum(0.0, fim - inicio), axis=2)\n    ag = np.prod(np.maximum(0.0, g[:, 2:] - g[:, :2]), axis=1)\n    ap = np.prod(np.maximum(0.0, p[:, 2:] - p[:, :2]), axis=1)\n    uniao = ag[:, None] + ap[None, :] - inter\n    return np.divide(inter, uniao, out=np.zeros_like(inter), where=uniao > 0)\n\n\ndef corresponder_objetos(gt, pred):\n    matriz = iou_matriz(gt, pred)\n    candidatos = []\n    for gi, g in enumerate(gt):\n        for pi, p in enumerate(pred):\n            if g["classe_id"] == p["classe_id"] and matriz[gi, pi] >= IOU_CORRESPONDENCIA:\n                candidatos.append((float(matriz[gi, pi]), gi, pi))\n    candidatos.sort(key=lambda item: (-item[0], item[1], item[2]))\n    usados_g, usados_p, pares = set(), set(), []\n    for iou, gi, pi in candidatos:\n        if gi not in usados_g and pi not in usados_p:\n            pares.append((gi, pi, iou))\n            usados_g.add(gi)\n            usados_p.add(pi)\n    return pares, [i for i in range(len(gt)) if i not in usados_g], [i for i in range(len(pred)) if i not in usados_p]\n\n\ndef contagem_eventos(gt, pred, largura, altura):\n    """Um FN/FP pode decorrer de detecção ausente/extra ou mudança de prioridade."""\n    g = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in gt]\n    p = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in pred]\n    pares, faltantes, extras = corresponder_objetos(g, p)\n    eventos = [(g[gi], p[pi]) for gi, pi, _ in pares]\n    eventos += [(g[gi], None) for gi in faltantes]\n    eventos += [(None, p[pi]) for pi in extras]\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for ref, prev in eventos:\n        a = bool(ref and ref["prioridade"] in {"medio", "alto"})\n        b = bool(prev and prev["prioridade"] in {"medio", "alto"})\n        contagem["tp"] += int(a and b)\n        contagem["fn"] += int(a and not b)\n        contagem["fp"] += int(not a and b)\n        contagem["alta_sem_alerta"] += int(bool(ref and ref["prioridade"] == "alto" and not b))\n    return contagem\n\n\ndef avaliar_alertas(modelo, registros, nomes, device=0):\n    modelo_nomes = {int(k): str(v) for k, v in modelo.names.items()}\n    if modelo_nomes != nomes:\n        raise ValueError("Mapa de classes do modelo difere do projeto.")\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for numero, r in enumerate(registros, 1):\n        resultado = modelo.predict(str(r["imagem"]), conf=CONF_ALERTA, imgsz=IMGSZ,\n                                   iou=IOU_NMS, max_det=MAX_DETECCOES,\n                                   device=device, verbose=False, save=False)[0]\n        caixas = resultado.boxes\n        pred = [] if caixas is None else [\n            {"classe_id": int(c), "classe": nomes[int(c)], "bbox": [float(v) for v in b]}\n            for c, b in zip(caixas.cls.cpu().numpy(), caixas.xyxy.cpu().numpy())]\n        eventos = contagem_eventos(r["gt"], pred, r["largura"], r["altura"])\n        for k, v in eventos.items():\n            contagem[k] += v\n        if numero % 100 == 0:\n            print(f"Validação dos alertas: {numero}/{len(registros)}")\n    tp, fn, fp = contagem["tp"], contagem["fn"], contagem["fp"]\n    dividir = lambda n, d: n / d if d else None\n    return {"imagens": len(registros), **contagem,\n            "precision_alertas": dividir(tp, tp + fp),\n            "recall_alertas": dividir(tp, tp + fn),\n            "f1_alertas": dividir(2 * tp, 2 * tp + fp + fn),\n            "referencia": "V2 fixa nas anotações; não valida risco físico independentemente",\n            "conf": CONF_ALERTA, "iou_nms": IOU_NMS,\n            "iou_correspondencia": IOU_CORRESPONDENCIA}\n'}
CODIGO_SHA256={'conf14_data.py': 'f73232836278c08edf0f955ae8fe50a33cb1d4eabe2b2cc35a5f410f095bae66', 'conf14_evaluation.py': '3e64c612ffcd7cb75fdf05707affc45cafa91acd1882ab311425c2752ed01656', 'cons19_demo.py': '33dfd31ef9c11b43058456fad522873fd8e0a8772a2eae17022d26ab141f45e8', 'cons19_inputs.py': 'ed81022938e964f027da3b9fdb691eb8f9782bfedd6bdb81ad19985bbb900e6c', 'cons19_io.py': '5049f9ba53b8df7c9dbef0c9b195e61cc6169dd99cd4e1c9dec6cbfbda05371e', 'cons19_protocol.py': 'd55fc25fe843a52004bb461e14612f7f116dd2019e0c721973ba2919fb5e2588', 'cons19_reports.py': 'e3900099c8ba98354d887b4d1ca55fe9b9e0bc9c3b7f5a410ea11f5aba316005', 'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/politica_alertas.py': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'src/validacao_alertas.py': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
PASTA_CODIGO=PASTA_LOCAL/'codigo';PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    if hashlib.sha256(fonte.encode()).hexdigest()!=CODIGO_SHA256[nome]:raise RuntimeError('Código embutido alterado: '+nome)
    compile(fonte,nome,'exec')
    for raiz in [PASTA_CODIGO,PASTA_RESULTADOS/'codigo']:
        destino=raiz/nome;destino.parent.mkdir(parents=True,exist_ok=True);destino.write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import cons19_protocol as protocolo19
import cons19_inputs as entradas19
import cons19_demo as demo19
import cons19_reports as relatorios19
import cons19_io as arquivos19
from src import politica_alertas as politica
from src import validacao_alertas as associacao
if core.sha256_file(politica.__file__)!=EXPECTED['politica'] or core.sha256_file(associacao.__file__)!=EXPECTED['associador']:
    raise RuntimeError('Política ou associador diferentes do protocolo.')
def progresso(etapa):
    global ULTIMA_ETAPA
    agora=time.monotonic();TEMPOS.append({'etapa':etapa,'segundos':agora-ULTIMA_ETAPA,'acumulado_segundos':agora-INICIO});ULTIMA_ETAPA=agora
    core.save_json(PASTA_RESULTADOS/'tempos_execucao_19.json',{'exclui_pip':True,'etapas':TEMPOS})
    print(f'{etapa} | {(agora-INICIO)/60:.1f} min acumulados',flush=True)
def registrar_falha(exc,etapa):
    pacote=arquivos19.failure(PASTA_RESULTADOS,exc,etapa,DRIVE,'resultados/'+NOME_EXPERIMENTO)
    print('Execução interrompida; pacote com o motivo:',pacote,flush=True);display(FileLink(str(pacote)))
progresso('Código e política conferidos')


## 3. Recuperar os pacotes 16–18 e as previsões registradas

Confere os resultados exatos já analisados. O pacote 17 fornece o GT e as dimensões
das validações. O 18 fornece as previsões compactas do baseline e do candidato 10.
Não será necessário baixar pesos, lotes de treino ou caches completos em 0,001.


In [ ]:
try:
    DRIVE=arquivos19.Drive.from_kaggle()
    if shutil.disk_usage(PASTA_LOCAL).free<10*1024**3:raise RuntimeError('Reserve 10 GiB livres para os ZIPs de entrada.')
    ENTRADAS=PASTA_LOCAL/'entradas'
    PACOTES={}
    for numero in [16,17,18]:
        arquivo=f'pacote_calibracao_{numero}.zip' if numero==17 else f'pacote_experimento_{numero}.zip'
        PACOTES[numero]=DRIVE.fetch(EXPECTED[f'pasta_{numero}']+'/'+arquivo,ENTRADAS/arquivo,EXPECTED[f'pacote_{numero}'])
    BUNDLE=entradas19.load(PACOTES,EXPECTED,PASTA_RESULTADOS)
    PLANO_FINAL=protocolo19.final_protocol(EXPECTED)
    CONTEXTO={'spec':protocolo19.SPEC,'entradas':{str(n):EXPECTED['pacote_'+str(n)] for n in [16,17,18]},
        'codigo_sha256':CODIGO_SHA256,'protocolo_final':PLANO_FINAL,'ambiente':AMBIENTE}
    CONTEXT_KEY=protocolo19.digest(CONTEXTO)
    NOME_EXPERIMENTO='consolidacao19_'+CONTEXT_KEY[:24]
    PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
    PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True);shutil.copytree(PASTA_BOOT,PASTA_RESULTADOS,dirs_exist_ok=True)
    core.save_json(PASTA_RESULTADOS/'contexto_execucao_19.json',{'context_key':CONTEXT_KEY,**CONTEXTO})
    core.save_json(PASTA_RESULTADOS/'protocolo_avaliacao_final.json',PLANO_FINAL)
    core.save_json(PASTA_RESULTADOS/'registro_protocolo_final.json',{'fixado_em_utc':datetime.now(timezone.utc).isoformat(),
        'sha256_canonico':protocolo19.digest(PLANO_FINAL),'reserva_consultada':False,'execucao_reserva_neste_notebook':False})
    progresso('Pacotes autenticados e protocolo posterior fixado')
    DRIVE.backup(PASTA_RESULTADOS,'resultados/'+NOME_EXPERIMENTO,required=False)
except Exception as exc:
    registrar_falha(exc,'pacotes e protocolo');raise


## 4. Consolidar métricas e escolher as imagens da demonstração

Reaplica V2 e associação às previsões em 0,25 para conferir as quatro referências,
sem inferência. Gera as tabelas completas dos experimentos anteriores.

Seleciona até 12 imagens por coorte, com regras e semente fixas: diferenças de
recuperação entre modelos, omissões altas, falsos alertas, acertos e imagens sem
alerta de referência nas classes avaliáveis. A galeria não altera as métricas nem
escolhe novos pesos ou limiares. O protocolo posterior já está fixado.


In [ ]:
try:
    AVALIACOES,CHECAGENS=entradas19.reproduce(BUNDLE,politica,associacao.corresponder_objetos,PASTA_RESULTADOS)
    TABELAS,FINAIS,HIPOTESES=relatorios19.consolidate(BUNDLE,PASTA_RESULTADOS)
    TODOS_REGISTROS=[r for c in protocolo19.COHORTS for r in BUNDLE['cohorts'][c]]
    SELECIONADAS,SELECAO=demo19.select(TODOS_REGISTROS,AVALIACOES)
    core.save_json(PASTA_RESULTADOS/'selecao_galeria_19.json',{'spec':protocolo19.SPEC,'selecionadas':SELECAO,
        'sha256_canonico':protocolo19.digest(SELECAO),'metricas_usam_toda_validacao':True})
    progresso('Tabelas prontas e casos da demonstração selecionados')
    print('Imagens ilustrativas:',len(SELECIONADAS),'| reserva permanece fechada.')
except Exception as exc:
    registrar_falha(exc,'métricas e seleção ilustrativa');raise


## 5. Baixar os ZIPs e extrair apenas as imagens ilustrativas

Os bytes e as dimensões de cada imagem selecionada devem corresponder ao cache.
Somente JPEGs/PNGs de Home VAL e da VAL externa serão abertos. O código não lê
labels ou coberturas de TEST/reserva e não executa extração integral dos ZIPs.


In [ ]:
try:
    ZIP_HOME=DRIVE.fetch('datasets/homeobjects_sunrgbd_validado.zip',ENTRADAS/'home.zip',EXPECTED['home_zip'])
    ZIP_NOVO=DRIVE.fetch(EXPECTED['pasta_validacao']+'/'+EXPECTED['arquivo_validacao'],ENTRADAS/'validacao_11.zip',EXPECTED['zip_novo'])
    ARQUIVOS_IMAGENS=demo19.extract_images(SELECIONADAS,{'home_val_historica':ZIP_HOME,'openimages_val_nova':ZIP_NOVO},EXPECTED,PASTA_RESULTADOS)
    progresso('Imagens da demonstração conferidas')
except Exception as exc:
    registrar_falha(exc,'imagens da demonstração');raise


## 6. Gerar demonstração, texto do TCC e pacote final

O HTML da demonstração é autossuficiente: mostra as três visões da mesma imagem,
permite filtrar por coorte e exibir todas as classes, e lista as mensagens V2.
Classes OI sem cobertura ficam marcadas sem julgamento de acerto/erro.

O relatório traz a comparação consolidada. O texto-base contém os números e as
limitações para o capítulo experimental. A avaliação nas 300 imagens é apenas
planejada; esta etapa não a executa e não troca o modelo em uso.

Ao término, os resultados estão em **pacote_consolidacao_19.zip**. Para explorar a demonstração,
baixe **demonstracao_19.html** e abra no navegador do seu computador.


In [ ]:
try:
    DEMONSTRACAO=demo19.gallery(SELECIONADAS,SELECAO,ARQUIVOS_IMAGENS,BUNDLE,AVALIACOES,politica,PASTA_RESULTADOS)
    RESUMO=relatorios19.finish(BUNDLE,AVALIACOES,CHECAGENS,SELECAO,PLANO_FINAL,CONTEXT_KEY,PASTA_RESULTADOS)
    if protocolo19.digest(json.loads((PASTA_RESULTADOS/'protocolo_avaliacao_final.json').read_text()))!=protocolo19.digest(PLANO_FINAL):
        raise RuntimeError('Protocolo posterior foi alterado durante a consolidação.')
    for nome,sha in CODIGO_SHA256.items():
        if core.sha256_file(PASTA_CODIGO/nome)!=sha:raise RuntimeError('Código mudou durante a execução: '+nome)
    core.save_json(PASTA_RESULTADOS/'integridade_final_19.json',{'conferida':True,'context_key':CONTEXT_KEY,
        'imagens_demonstracao':len(SELECAO),'reproducao_referencias':all(r['reproduziu'] for r in CHECAGENS),
        'treinamento_executado':False,'inferencia_executada':False,'reserva_extraida':False,'reserva_inferida':False,
        'test_home_inferido':False,'test_oi20k_inferido':False,'modelo_substituido':False,
        'protocolo_final_sha256':protocolo19.digest(PLANO_FINAL)})
    anterior=PASTA_RESULTADOS/'execucao_interrompida_19.json'
    if anterior.exists():
        historico=PASTA_RESULTADOS/'historico_falhas';historico.mkdir(exist_ok=True)
        anterior.replace(historico/('falha_resolvida_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.json'))
    progresso('Demonstração, relatório e protocolo concluídos')
    PACOTE_FINAL=arquivos19.package(PASTA_RESULTADOS)
    COPIADO=DRIVE.backup(PASTA_RESULTADOS,'resultados/'+NOME_EXPERIMENTO,required=False)
    print('NOTEBOOK 19 CONCLUÍDO — demonstração, tabelas, texto-base e protocolo posterior gerados.')
    if not COPIADO:print('A cópia ao Drive falhou; baixe os arquivos na saída do Kaggle. Execute esta última célula para repetir a cópia.')
    print('Arquivo de resultados:',PACOTE_FINAL)
    for p in [PACOTE_FINAL,DEMONSTRACAO,PASTA_RESULTADOS/'relatorio_consolidacao_19.html',
              PASTA_RESULTADOS/'texto_base_capitulo4.md',PASTA_RESULTADOS/'protocolo_avaliacao_final.json']:
        display(FileLink(str(p)))
except Exception as exc:
    registrar_falha(exc,'demonstração e pacote final');raise
